# Integrating Irregularly Sampled Data

## From quadrature to modeling

For a known function, a quadrature rule chooses where to evaluate that function. Experimental data present a different problem: values are already available at a finite—and often irregular—set of locations. Integrating them requires an assumption about what happens **between** measurements.

The composite trapezoid rule on nonuniform points $(x_i,y_i)$ is

$$I_T=\sum_{i=0}^{N-1}\frac{y_i+y_{i+1}}{2}(x_{i+1}-x_i).$$

This is exactly the integral of the piecewise-linear interpolant through the data. Nearest-neighbor, quadratic, and cubic interpolation represent different models between points and can therefore produce different integrals even when the final quadrature grid is extremely fine.

This notebook separates three sources of disagreement:

1. **sampling error:** important features may lie between the measured points;
2. **interpolation/model error:** different assumptions connect those points differently; and
3. **quadrature error:** the chosen interpolant is itself integrated with a finite numerical grid.

## Synthetic benchmark: a sum of Gaussian components

We create an underlying function from three normalized Gaussians,

$$g(x;\mu,\sigma,A)=\frac{A}{\sqrt{2\pi}\sigma}
\exp\left[-\frac{(x-\mu)^2}{2\sigma^2}\right].$$

Here $\sigma$ is the standard deviation and $A$ is the **total area** of the Gaussian, not its peak height. Between finite limits,

$$\int_{x_1}^{x_2}g(x)\,dx=
\frac{A}{2}\left[\operatorname{erf}\left(\frac{x_2-\mu}{\sqrt{2}\sigma}\right)
-\operatorname{erf}\left(\frac{x_1-\mu}{\sqrt{2}\sigma}\right)\right].$$

Because the generating function is known, its analytical integral provides a benchmark that would normally be unavailable for real measurements.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import special
from scipy.interpolate import interp1d


# (centroid, standard deviation, total area)
gaussian_components = (
    (0.40, 3.0, 500.0),
    (10.0, 7.0, 1000.0),
    (40.0, 15.0, 3000.0),
)


def gaussian(x, centroid, sigma, area):
    '''Normalized Gaussian whose integral over the real line equals area.'''
    return area/(np.sqrt(2.0*np.pi)*sigma)*np.exp(
        -0.5*((np.asarray(x)-centroid)/sigma)**2
    )


def generating_function(x):
    '''Sum of the three Gaussian components.'''
    return sum(gaussian(x, *parameters) for parameters in gaussian_components)


def gaussian_integral(xlow, xhigh, centroid, sigma, area):
    upper = (xhigh-centroid)/(np.sqrt(2.0)*sigma)
    lower = (xlow-centroid)/(np.sqrt(2.0)*sigma)
    return 0.5*area*(special.erf(upper)-special.erf(lower))


def analytic_integral(xlow, xhigh):
    return sum(
        gaussian_integral(xlow, xhigh, *parameters)
        for parameters in gaussian_components
    )


def composite_trapezoid(x, y):
    '''Composite trapezoid rule for a strictly increasing, nonuniform grid.'''
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.ndim != 1 or y.ndim != 1 or len(x) != len(y):
        raise ValueError("x and y must be one-dimensional arrays of equal length")
    if len(x) < 2 or np.any(np.diff(x) <= 0):
        raise ValueError("x must contain at least two strictly increasing values")
    return np.sum(0.5*(y[:-1]+y[1:])*np.diff(x))

## Sparse, irregularly spaced observations

The array below represents the locations at which the synthetic function was “measured.” The large and uneven gaps are intentional. A dense grid is used only to display the hidden generating function and to demonstrate that trapezoid quadrature becomes extremely accurate when the underlying function is densely sampled.

For real experimental data, the red generating curve and analytical answer would not be known. They are included here solely to evaluate the numerical and modeling choices.

In [ ]:
x_data = np.array([
    0.1, 0.2, 0.45, 0.92, 2.7, 4.9, 5.6, 5.7,
    9.2, 10.4, 20.2, 40.5, 41.5, 44.3, 60.2,
])
y_data = generating_function(x_data)

x_fine = np.linspace(x_data[0], x_data[-1], 10_000)
y_fine = generating_function(x_fine)

xlow, xhigh = x_data[0], x_data[-1]
exact = analytic_integral(xlow, xhigh)
dense_trapezoid = composite_trapezoid(x_fine, y_fine)
data_trapezoid = composite_trapezoid(x_data, y_data)

print(f"Analytical integral:                 {exact:.9f}")
print(f"Dense-grid trapezoid:               {dense_trapezoid:.9f} "
      f"({100*(dense_trapezoid-exact)/exact:+.4f}%)")
print(f"Trapezoid on irregular observations: {data_trapezoid:.9f} "
      f"({100*(data_trapezoid-exact)/exact:+.4f}%)")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_fine, y_fine, 'r--', label='Hidden generating function')
ax.plot(x_data, y_data, 'o', label='Irregular observations')
ax.plot(x_data, y_data, '-', label='Piecewise-linear interpolant')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title('Irregular Sampling and the Trapezoid Rule')
ax.legend()
plt.show()

## Comparing interpolation models

We now construct nearest-neighbor, linear, quadratic, and cubic interpolants through exactly the same observations. Each interpolant is evaluated on the same fine grid and integrated with the same trapezoid routine.

Because the fine grid is dense, differences among the resulting integrals are dominated by the **interpolation model**, not by the final trapezoid calculation. In particular:

- integrating the linear interpolant reproduces the direct nonuniform trapezoid result;
- nearest-neighbor interpolation is a piecewise-constant model, not the midpoint quadrature rule; and
- higher-order interpolation is not automatically more accurate—it can curve or overshoot between widely separated observations.

In [ ]:
interpolants = {
    'nearest': interp1d(x_data, y_data, kind='nearest'),
    'linear': interp1d(x_data, y_data, kind='linear'),
    'quadratic': interp1d(x_data, y_data, kind='quadratic'),
    'cubic': interp1d(x_data, y_data, kind='cubic'),
}

interpolated_integrals = {}
interpolated_errors = {}
for name, interpolant in interpolants.items():
    estimate = composite_trapezoid(x_fine, interpolant(x_fine))
    interpolated_integrals[name] = estimate
    interpolated_errors[name] = 100.0*(estimate-exact)/exact
    print(f"{name:9s}: integral={estimate:.9f}, "
          f"relative error={interpolated_errors[name]:+.4f}%")

print(f"\nLinear interpolant minus direct data trapezoid: "
      f"{interpolated_integrals['linear']-data_trapezoid:.3e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for axis in axes:
    axis.plot(x_fine, y_fine, 'k--', label='Hidden generating function')
    axis.plot(x_data, y_data, 'o', label='Observations')
    axis.set_xlabel('x')
    axis.set_ylabel('f(x)')

for name in ('nearest', 'linear'):
    axes[0].plot(x_fine, interpolants[name](x_fine),
                 label=f"{name}: error={interpolated_errors[name]:+.3f}%")
axes[0].set_title('Piecewise-constant and Linear Models')
axes[0].legend()

for name in ('quadratic', 'cubic'):
    axes[1].plot(x_fine, interpolants[name](x_fine),
                 label=f"{name}: error={interpolated_errors[name]:+.3f}%")
axes[1].set_title('Higher-order Interpolation Models')
axes[1].legend()

fig.suptitle('How the Interpolation Model Changes the Integral')
plt.tight_layout()
plt.show()

## Conclusions

When integrating tabulated data, the trapezoid rule is more than an arithmetic formula: it asserts that the function is linear between neighboring observations. A different interpolation method makes a different modeling assumption and can change the integral even when every method passes through the same measured points.

For real data, interpolation order should be chosen using knowledge of the process, measurement spacing, noise, and validation—not simply by selecting the smoothest-looking curve. Numerical precision on the chosen interpolant does not measure uncertainty in that underlying model.